# LangGraph：把 Agent 写成可控制的状态图

LangGraph 的核心不是 Prompt，而是 **State（状态） + Node（步骤） + Edge（流转）**。

适合：审批流、长任务、需要暂停/恢复、需要明确分支、需要审计轨迹的业务。

官方文档：https://docs.langchain.com/oss/python/langgraph/overview

项目对应实现：`support_project/adapters/langgraph_adapter.py`


In [ ]:
from pathlib import Path
from langchain_cases.common import setup_utf8
from langchain_cases.config import get_settings

setup_utf8()

s = get_settings()
print("模型:", s.chat_model)
print("项目根目录:", Path.cwd())


## 1. 最小 StateGraph

三个节点按固定顺序执行，`Annotated[list[str], operator.add]` 让多个节点追加 trace，
而不是互相覆盖状态。

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

class DemoState(TypedDict):
    text: str
    trace: Annotated[list[str], operator.add]

def normalize(state: DemoState):
    return {"text": state["text"].strip().lower(), "trace": ["normalize"]}

def enrich(state: DemoState):
    return {"text": f"processed:{state['text']}", "trace": ["enrich"]}

builder = StateGraph(DemoState)
builder.add_node("normalize", normalize)
builder.add_node("enrich", enrich)
builder.add_edge(START, "normalize")
builder.add_edge("normalize", "enrich")
builder.add_edge("enrich", END)
app = builder.compile()

print(app.invoke({"text": "  Hello LangGraph  ", "trace": []}))


## 2. 售后项目：显式流程 + 一次 LLM 生成

项目把查询、政策检索、退款计算做成确定性节点，最后才调用模型组织语言。
这比“完全让模型决定步骤”更适合需要审计的业务。

In [ ]:
from support_project.adapters.langgraph_adapter import build_graph

graph = build_graph()
result = graph.invoke(
    {
        "question": "订单 A1001 的降噪耳机坏了，想退款，是否符合政策？",
        "trace": [],
    },
    config={"configurable": {"thread_id": "notebook-langgraph-demo"}},
)
print("TRACE:", result["trace"])
print("\nANSWER:\n", result["answer"])


## 3. 原生 Human-in-the-loop（可暂停、可恢复）

`interrupt()` 会把图暂停并通过 checkpointer 保存状态；外部审批后用
`Command(resume=...)` 继续。下面的代码会自动模拟一次审批。

In [ ]:
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt

class ApprovalState(TypedDict):
    result: str

def approval_node(state: ApprovalState):
    # 第一次执行到这里会暂停；恢复时 interrupt() 返回 resume 值。
    decision = interrupt({"question": "是否批准这笔退款？"})
    return {"result": f"审批结果：{decision}"}

b = StateGraph(ApprovalState)
b.add_node("approval", approval_node)
b.add_edge(START, "approval")
b.add_edge("approval", END)
approval_app = b.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "notebook-hitl-demo"}}

first = approval_app.invoke({"result": ""}, config=config)
print("暂停状态：", first["__interrupt__"][0].value)

second = approval_app.invoke(Command(resume="approved"), config=config)
print("恢复结果：", second)


## 4. 状态历史、时间旅行和流程图可视化

这是 LangGraph 相比普通 Agent 最“工程化”的地方：

- 每个节点执行后都会形成一个 checkpoint；
- `get_state_history()` 可以倒序查看每一步的 `values`、`next` 和 `checkpoint_id`；
- 用旧 checkpoint 的 `config` 调用 `invoke(None, config)` 可以从那里重放；
- `update_state()` 可以基于旧 checkpoint 创建 fork，走另一条分支；
- `get_graph().draw_mermaid()` 可以把流程拓扑导出成 Mermaid 图。

注意：**Replay 不是读缓存**，checkpoint 之后的节点会重新执行，LLM 和外部 API 也会再次调用。

In [ ]:
from IPython.display import Markdown, display
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END

class TraceState(TypedDict):
    value: int
    steps: list[str]

def add_one(state: TraceState):
    return {"value": state["value"] + 1, "steps": state["steps"] + ["add_one"]}

def double(state: TraceState):
    return {"value": state["value"] * 2, "steps": state["steps"] + ["double"]}

trace_builder = StateGraph(TraceState)
trace_builder.add_node("add_one", add_one)
trace_builder.add_node("double", double)
trace_builder.add_edge(START, "add_one")
trace_builder.add_edge("add_one", "double")
trace_builder.add_edge("double", END)
trace_graph = trace_builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "notebook-time-travel"}}
print("初始运行：", trace_graph.invoke({"value": 1, "steps": []}, config))

history = list(trace_graph.get_state_history(config))
print("\n历史 checkpoint（新 → 旧）：")
for snapshot in history:
    print(
        f"step={snapshot.metadata.get('step')} "
        f"next={snapshot.next} values={snapshot.values}"
    )

# 找到 double 执行前的 checkpoint，从那里重放。
before_double = next(item for item in history if item.next == ("double",))
print("\n从 double 之前重放：", trace_graph.invoke(None, before_double.config))

# 从同一个 checkpoint 分叉：把 value 改成 10，再继续执行 double。
fork_config = trace_graph.update_state(
    before_double.config,
    {"value": 10},
    as_node="add_one",
)
print("fork 后结果：", trace_graph.invoke(None, fork_config))

print("\n售后项目的流程拓扑：")
display(Markdown(f"```mermaid\n{build_graph().get_graph().draw_mermaid()}\n```"))


## 何时选 LangGraph

- 业务流程本身就是核心复杂度，而不是“调用哪个工具”；
- 需要固定步骤、条件分支、并行、子图或人工审批；
- 需要每个步骤可观察、可恢复、可回放；
- 需要把 LangChain、LlamaIndex、普通 Python 服务放进同一张图。

项目入口：

```powershell
uv run python -m support_project.runner --framework langgraph --full
```
